In [1]:
import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

### Генерация датасетов с использованием RandomOverSampler

In [2]:
track_cities = [
        'Melbourne', 'Shanghai', 'Suzuka', 'Sakhir', 'Jeddah', 
        'Miami', 'Imola', 'Monaco', 'Barcelona', 'Montreal', 
        'Spielberg', 'Silverstone', 'Spa', 'Budapest', 'Zandvoort', 
        'Monza', 'Baku', 'Singapore', 'Austin', 'Mexico City', 
        'Sao Paulo', 'Las Vegas', 'Lusail', 'Abu Dhabi'
    ]

In [3]:
def get_avg_speed(n):   # количественный    
    return np.random.uniform(50, 350, n)

def get_wins_count(n):  
    return np.random.randint(0, 100, n)

def get_number_of_overtaking_last_year(n):
    return np.random.randint(0, 100, n)

def get_driver_salary_mln(n):
    return np.random.randint(1, 66, n)

def get_dnf_percent(n):
    return np.random.randint(0, 30, n)

def get_penalty_points(n):
    return np.random.randint(0, 12, n)

def get_personal_position(n):       # порядковый
    return np.random.randint(1, 21, n)

def get_rain_forecast(n):    # бинарный
    return np.random.randint(0, 2, n)

def get_team_name(n):    # номинальный
    team_names = ['Alpine', 'Aston Martin', 'Williams', 'Sauber', 'Ferrari', 
                  'Haas', 'McLaren', 'Mercedes', 'Racing Bulls', 'Red Bull']
    return np.random.choice(team_names, n)

def get_pilot_name(n):    
    pilot_names = ['Gasly', 'Colapinto', 'Alonso', 'Stroll', 'Sainz', 'Albon', 
                   'Hulkenberg', 'Bortoleto', 'Leclerc', 'Hamilton', 'Ocon', 
                   'Bearman', 'Norris', 'Piastri', 'Russel', 'Antonelli', 
                   'Lawson', 'Tsunoda', 'Verstappen', 'Hadjar']
    return np.random.choice(pilot_names, n)

def get_track_city(n):    
    return np.random.choice(track_cities, n)

In [4]:
feature_pool = [   
        ('pilot_name', get_pilot_name),
        ('team_name', get_team_name),
        ('track_city', get_track_city),
        ('dnf_percent', get_dnf_percent),
        ('avg_speed', get_avg_speed), 
        ('wins_count', get_wins_count),
        ('number_of_overtaking_last_year', get_number_of_overtaking_last_year),
        ('driver_salary_mln', get_driver_salary_mln),
        ('penalty_points', get_penalty_points),
        ('personal_position', get_personal_position),
        ('rain_forecast', get_rain_forecast),          
    ]

In [5]:
pilot_to_team = {
    'Verstappen': 'Red Bull',
    'Tsunoda': 'Red Bull Racing', 
    
    'Hamilton': 'Ferrari',
    'Leclerc': 'Ferrari',
    
    'Norris': 'McLaren',
    'Piastri': 'McLaren',
    
    'Russel': 'Mercedes',
    'Antonelli': 'Mercedes',
    
    'Alonso': 'Aston Martin',
    'Stroll': 'Aston Martin',
    
    'Gasly': 'Alpine',
    'Colapinto': 'Alpine',
    
    'Sainz': 'Williams',
    'Albon': 'Williams',
    
    'Lawson': 'Racing Bulls',
    'Hadjar': 'Racing Bulls',
    
    'Bearman': 'Haas',
    'Ocon': 'Haas',

    'Hulkenberg': 'Sauber',
    'Bortoleto': 'Sauber'
}


In [6]:
def calculate_dnf_collision(row):
    dnf_risk = (row['dnf_percent'] / 100) if 'dnf_percent' in row else 0.20
    avg_speed = row.get('avg_speed')
    rain_forecast = row['rain_forecast'] if 'rain_forecast' in row else 0
    personal_position = row['personal_position'] if 'personal_position' in row else 10
    wins_count = row['wins_count'] if 'wins_count' in row else 0
    number_of_overtaking = row['number_of_overtaking_last_year'] if 'number_of_overtaking_last_year' in row else 0
    penalty_points = row['penalty_points'] if 'penalty_points' in row else 0

    total_probility = dnf_risk

    if avg_speed < 150:
        total_probility += 0.5

    if rain_forecast == 1:
        total_probility += 0.4

    if penalty_points > 5:
        total_probility += 0.2   
    
    if personal_position > 10:
        total_probility += 0.2
    else:
        total_probility -= 0.1

    if wins_count > 30 and number_of_overtaking > 100:
        total_probility -= 0.3
    elif wins_count < 10 and number_of_overtaking > 60:
        total_probility += 0.2
    else:
        total_probility -= 0.05

    total_probility = max(0, min(total_probility, 1))

    if total_probility > 0.3:
        return 1
    
    return 0

In [7]:
def generate_data(n_samples, m_features):
    
    
    selected_feats = feature_pool[:m_features]
    data = {}
    
    for feat, func in selected_feats:
        data[f'{feat}'] = func(n_samples)

    f1_data = pd.DataFrame(data)

    if 'pilot_name' in f1_data.columns and 'team_name' in f1_data.columns:
        f1_data['team_name'] = f1_data['pilot_name'].map(pilot_to_team)

    f1_data['collision'] = f1_data.apply(calculate_dnf_collision, axis=1)

    return f1_data

In [8]:
n_samples = [100, 500, 1000, 3000]
m_features = [5, 8, 11]

np.random.seed(81)
ros = RandomOverSampler(random_state=81)
for n in n_samples:
    for m in m_features:
        data = generate_data(n, m)
        x = data.drop('collision', axis=1)
        y = data['collision']

        if y.nunique() == 2:
            x_resampled, y_resampled = ros.fit_resample(x, y)
            data = pd.concat([pd.DataFrame(x_resampled), pd.Series(y_resampled, name='collision')], axis=1)
            
        data.to_csv(f"f1_data/f1_data_{n}_s_{m}_f.csv", index=False)

In [9]:
test = pd.read_csv('f1_data/f1_data_3000_s_10_f.csv')

print("Распределение классов:")
test['collision'].value_counts()

Распределение классов:


collision
0    1788
1    1788
Name: count, dtype: int64

### Алгоритмы машинного обучения

![Scikit-learn Map](ml_map.png)

-  Имеется > 50 записей 
- Предсказываем категорию (1 - сход пилота, 0 - нет)
- Существует столбец `collision`
- Записей < 100k, 

следовательно, согласно схеме библиотеки `sckit-learn` следует выбрать в качетсве основной модели `Linear SVC (Linear Support Vector Classification)` - метод опорных векторов. 

Определение возможности схода пилота является задачей классификации, так как нужно каждой записи присвоить метку. Будем рассматривать розовую область `classification`.

### Итоговый список рассматриваемых алгоритмов

Изучим полученные метрики следующих алгоритмов:
- **Linear SVC** - линейный классификатор, который ищет оптимальную разделяющую гиперплоскость с максимальным зазором между классами.
- **Native Bayes** - основан на теореме Байеса с «наивным» предположением о независимости признаков
- **KNeighbors Classifier** - относит объект к тому классу, который наиболее часто встречается среди k его ближайших соседей в пространстве признаков
- **Ensemble Classifiers (Random Forest)** - cоздает ансамбль из множества независимых деревьев решений и усредняет их предсказания.

### Кодирование категориальных признаков

In [10]:
pilot_names = pilot_to_team.keys()
pilot_mapping = {name: i for i, name in enumerate(pilot_names)}

teams = pilot_to_team.values()
teams_mapping = {team: i for i, team in enumerate(teams)}

track_cities_mapping = {track : i for i, track in enumerate(track_cities)}

for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"f1_data/f1_data_{n}_s_{m}_f.csv")

        data['pilot_name'] = data['pilot_name'].map(pilot_mapping)
        data['team_name'] = data['team_name'].map(teams_mapping)
        data['track_city'] = data['track_city'].map(track_cities_mapping)
        
        data.to_csv(f"f1_data/f1_data_{n}_s_{m}_f.csv", index=False)